In [ ]:

from pyspark.sql import functions as F

storage = "sthybridlakeprojectdev.dfs.core.windows.net"

silver_path = f"abfss://silver@{storage}/sqlserver/calendar/"
gold_path = f"abfss://gold@{storage}/dim_date/"

# Read Silver Calendar
calendar_df = spark.read.format("delta").load(silver_path)

# Build Gold Date Dimension
dim_date_df = (
    calendar_df
    .select("Date")
    .withColumn(
        "DateKey",
        F.date_format("Date", "yyyyMMdd").cast("int")
    )
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("Day", F.dayofmonth("Date"))
    .withColumn("DayOfWeek", F.dayofweek("Date"))
    .withColumn("DayName", F.date_format("Date", "EEEE"))
    .withColumn(
        "IsWeekend",
        F.dayofweek("Date").isin(1, 7)
    )
)

# Data quality validation
date_count = dim_date_df.count()
duplicate_keys = (
    dim_date_df.groupBy("DateKey")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
null_keys = dim_date_df.filter(
    F.col("DateKey").isNull()
).count()

print("Date records:", date_count)
print("Duplicate DateKeys:", duplicate_keys)
print("Null DateKeys:", null_keys)

assert date_count == 1095, "Unexpected date count"
assert duplicate_keys == 0, "Duplicate DateKey found"
assert null_keys == 0, "Null DateKey found"

# Write Gold Delta
(
    dim_date_df.write
    .format("delta")
    .mode("overwrite")
    .save(gold_path)
)

print("Gold dim_date PASSED")


In [ ]:

from pyspark.sql import functions as F

storage = "sthybridlakeprojectdev.dfs.core.windows.net"
silver = f"abfss://silver@{storage}"
gold = f"abfss://gold@{storage}"

# Read Silver datasets
product = spark.read.format("delta").load(
    f"{silver}/mysql/product/"
)

subcategory = spark.read.format("delta").load(
    f"{silver}/mysql/subcategory/"
)

category = spark.read.format("delta").load(
    f"{silver}/mysql/category/"
)

# Build product dimension
dim_product = (
    product.alias("p")
    .join(
        subcategory.alias("s"),
        F.col("p.ProductSubcategoryKey") ==
        F.col("s.ProductSubcategoryKey"),
        "left"
    )
    .join(
        category.alias("c"),
        F.col("s.ProductCategoryKey") ==
        F.col("c.ProductCategoryKey"),
        "left"
    )
    .select(
        F.col("p.ProductKey"),
        F.col("p.ProductName"),
        F.col("p.Manufacturer"),
        F.col("p.BrandName"),
        F.col("p.ClassName"),
        F.col("p.StyleName"),
        F.col("p.ColorName"),
        F.col("p.ProductSubcategoryKey"),
        F.col("s.ProductSubcategoryName"),
        F.col("s.ProductCategoryKey"),
        F.col("c.ProductCategoryName")
    )
)

# Validate dimension
source_count = product.count()
gold_count = dim_product.count()

duplicate_count = (
    dim_product.groupBy("ProductKey")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_key_count = dim_product.filter(
    F.col("ProductKey").isNull()
).count()

print("Silver Product records:", source_count)
print("Gold Product records:", gold_count)
print("Duplicate ProductKeys:", duplicate_count)
print("Null ProductKeys:", null_key_count)

assert source_count == gold_count, "Product count mismatch"
assert duplicate_count == 0, "Duplicate ProductKeys found"
assert null_key_count == 0, "Null ProductKeys found"

# Write Gold Delta
(
    dim_product.write
    .format("delta")
    .mode("overwrite")
    .save(f"{gold}/dim_product/")
)

print("Gold dim_product PASSED")


In [ ]:

from pyspark.sql import functions as F

storage = "sthybridlakeprojectdev.dfs.core.windows.net"
silver = f"abfss://silver@{storage}"
gold = f"abfss://gold@{storage}"

# Build Channel Dimension
channel = spark.read.format("delta").load(
    f"{silver}/sftp/channel/"
)

dim_channel = channel.select(
    "ChannelKey",
    "Channel"
)

# Build Promotion Dimension
promotion = spark.read.format("delta").load(
    f"{silver}/sftp/promotion/"
)

dim_promotion = promotion.select(
    "PromotionKey",
    "PromotionName",
    "PromotionDescription",
    "DiscountPercent",
    "PromotionType",
    "PromotionCategory"
)

# Validate dimensions before writing
for name, df, key, expected in [
    ("dim_channel", dim_channel, "ChannelKey", 4),
    ("dim_promotion", dim_promotion, "PromotionKey", 28)
]:
    row_count = df.count()

    duplicate_count = (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_count = df.filter(F.col(key).isNull()).count()

    print(f"{name}: rows={row_count}, duplicates={duplicate_count}, nulls={null_count}")

    assert row_count == expected, f"Unexpected row count: {name}"
    assert duplicate_count == 0, f"Duplicate keys: {name}"
    assert null_count == 0, f"Null keys: {name}"

# Write Gold dimensions
(
    dim_channel.write
    .format("delta")
    .mode("overwrite")
    .save(f"{gold}/dim_channel/")
)

(
    dim_promotion.write
    .format("delta")
    .mode("overwrite")
    .save(f"{gold}/dim_promotion/")
)

print("Gold dim_channel PASSED")
print("Gold dim_promotion PASSED")


In [ ]:

from pyspark.sql import functions as F
from functools import reduce

storage = "sthybridlakeprojectdev.dfs.core.windows.net"
silver = f"abfss://silver@{storage}"
gold = f"abfss://gold@{storage}"

# Read and combine three Silver transaction datasets
years = [2021, 2022, 2023]

transaction_dfs = [
    spark.read.format("delta").load(
        f"{silver}/sqlserver/transactions_{year}/"
    )
    for year in years
]

transactions = reduce(
    lambda left, right: left.unionByName(right),
    transaction_dfs
)

# Build fact_sales at the original source transaction-row grain
fact_sales = transactions.select(
    F.date_format("TransactionDate", "yyyyMMdd")
        .cast("int").alias("DateKey"),
    "ProductKey",
    "ChannelKey",
    "PromotionKey",
    "UnitCost",
    "UnitPrice",
    "SalesQuantity",
    "ReturnQuantity",
    "ReturnAmount",
    "DiscountQuantity",
    "DiscountAmount",
    "TotalCost",
    "SalesAmount",
    "LoadID",
    "ingestion_timestamp",
    "source_system",
    "source_object"
)

# Reconcile row counts
source_count = transactions.count()
fact_count = fact_sales.count()

# Validate mandatory keys and financial measures
invalid_count = fact_sales.filter(
    F.col("DateKey").isNull()
    | F.col("ProductKey").isNull()
    | F.col("ChannelKey").isNull()
    | F.col("PromotionKey").isNull()
    | F.col("SalesAmount").isNull()
    | F.col("SalesQuantity").isNull()
).count()

print("Silver transaction rows:", source_count)
print("Gold fact rows:", fact_count)
print("Invalid fact rows:", invalid_count)

assert source_count == 2798589, "Unexpected Silver transaction count"
assert fact_count == source_count, "Fact row count mismatch"
assert invalid_count == 0, "Invalid fact rows found"

# Write Gold Delta
(
    fact_sales.write
    .format("delta")
    .mode("overwrite")
    .save(f"{gold}/fact_sales/")
)

# Verify persisted data
written_count = spark.read.format("delta").load(
    f"{gold}/fact_sales/"
).count()

assert written_count == fact_count, "Persisted fact count mismatch"

print("Gold fact_sales PASSED")
print("Persisted rows:", written_count)


In [ ]:

from pyspark.sql import functions as F

gold = "abfss://gold@sthybridlakeprojectdev.dfs.core.windows.net"

fact = spark.read.format("delta").load(f"{gold}/fact_sales/")

relationships = {
    "DateKey": ("dim_date", "DateKey"),
    "ProductKey": ("dim_product", "ProductKey"),
    "ChannelKey": ("dim_channel", "ChannelKey"),
    "PromotionKey": ("dim_promotion", "PromotionKey")
}

for fact_key, (table_name, dim_key) in relationships.items():

    dim = spark.read.format("delta").load(
        f"{gold}/{table_name}/"
    )

    unmatched_rows = (
        fact.select(fact_key)
        .join(
            dim.select(F.col(dim_key).alias(fact_key)).distinct(),
            fact_key,
            "left_anti"
        )
        .count()
    )

    print(f"{table_name}: unmatched fact rows = {unmatched_rows}")

    assert unmatched_rows == 0, (
        f"Referential integrity failed: {table_name}"
    )

print("\nGOLD STAR SCHEMA VALIDATION PASSED")
